# Feature Extraction: eICU (train) & MIMIC-III (validation)

**Goal:** Extract a harmonized feature set from both datasets so that a model trained on eICU can be externally validated on MIMIC-III.

**Output files:**
- `eicu_features.csv` — training set
- `mimic_features.csv` — validation set

**Feature domains extracted:**
1. Demographics (age, gender, ethnicity)
2. ICU stay metadata (LOS, unit type, admission source)
3. First-24h lab summary stats (min / max / mean for 12 key labs)
4. First-24h vital signs summary stats
5. Diagnosis category (ICD-9 chapter)
6. Target label: ICU mortality (binary)

# 0. Mount Google Drive


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
EICU_DIR  = '/content/drive/MyDrive/AI in Medicine/data/eicu_raw/'
MIMIC_DIR = '/content/drive/MyDrive/AI in Medicine/data/mimic_3_raw/mimic/'
EICU_OUT_DIR   = '/content/drive/MyDrive/AI in Medicine/data/output_data/eicu_train/'
MIMIC_OUT_DIR   = '/content/drive/MyDrive/AI in Medicine/data/output_data/mimic_val/'


import os
os.makedirs(EICU_OUT_DIR, exist_ok=True)
os.makedirs(MIMIC_OUT_DIR, exist_ok=True)

print('Paths configured.')

Paths configured.


In [9]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
print('Libraries loaded.')

Libraries loaded.


# 1.1. eICU Feature Extraction


In [10]:
def load_eicu(fname, **kwargs):
    """Load eICU CSV."""
    path = os.path.join(EICU_DIR, fname)

    for ext in ['.csv']:
        full = path + ext if not path.endswith(ext) else path
        if os.path.exists(full):
            return pd.read_csv(full, **kwargs)
    raise FileNotFoundError(f'Cannot find {fname} in {EICU_DIR}')

# ── Patient (core demographics + stay info)
eicu_pt = load_eicu('patient', low_memory=False)
eicu_pt.columns = eicu_pt.columns.str.lower()
print(f'patient rows: {len(eicu_pt):,}  |  cols: {list(eicu_pt.columns)}')

patient rows: 2,520  |  cols: ['patientunitstayid', 'patienthealthsystemstayid', 'gender', 'age', 'ethnicity', 'hospitalid', 'wardid', 'apacheadmissiondx', 'admissionheight', 'hospitaladmittime24', 'hospitaladmitoffset', 'hospitaladmitsource', 'hospitaldischargeyear', 'hospitaldischargetime24', 'hospitaldischargeoffset', 'hospitaldischargelocation', 'hospitaldischargestatus', 'unittype', 'unitadmittime24', 'unitadmitsource', 'unitvisitnumber', 'unitstaytype', 'admissionweight', 'dischargeweight', 'unitdischargetime24', 'unitdischargeoffset', 'unitdischargelocation', 'unitdischargestatus', 'uniquepid']


In [11]:
# ── Lab results
eicu_lab = load_eicu('lab', low_memory=False)
eicu_lab.columns = eicu_lab.columns.str.lower()
print(f'lab rows: {len(eicu_lab):,}')
eicu_lab.head(3)

lab rows: 434,660


,labid,patientunitstayid,labresultoffset,labtypeid,labname,labresult,labresulttext,labmeasurenamesystem,labmeasurenameinterface,labresultrevisedoffset
0,437880563,1754323,-647,3,Hct,38.300,38.3,%,%,-631
1,437880572,1754323,-647,3,platelets x 1000,181.000,181,K/mcL,k/mm cu,-631
2,437880560,1754323,-647,3,RBC,4.860,4.86,M/mcL,m/mm cu,-631


In [12]:
# ── Vital signs (periodic = 5-min medians)
eicu_vp = load_eicu('vitalPeriodic', low_memory=False)
eicu_vp.columns = eicu_vp.columns.str.lower()
print(f'vitalPeriodic rows: {len(eicu_vp):,}')
print(eicu_vp.columns.tolist())

vitalPeriodic rows: 1,634,960
['vitalperiodicid', 'patientunitstayid', 'observationoffset', 'temperature', 'sao2', 'heartrate', 'respiration', 'cvp', 'etco2', 'systemicsystolic', 'systemicdiastolic', 'systemicmean', 'pasystolic', 'padiastolic', 'pamean', 'st1', 'st2', 'st3', 'icp']


In [13]:
# ── Diagnosis
eicu_dx = load_eicu('diagnosis', low_memory=False)
eicu_dx.columns = eicu_dx.columns.str.lower()
print(f'diagnosis rows: {len(eicu_dx):,}')
eicu_dx.head(3)

diagnosis rows: 24,978


,diagnosisid,patientunitstayid,activeupondischarge,diagnosisoffset,diagnosisstring,icd9code,diagnosispriority
0,7607199,346380,False,5028,cardiovascular|ventricular disorders|hypertension,"401.9, I10",Other
1,7570429,346380,False,685,neurologic|altered mental status / pain|change...,"780.09, R41.82",Major
2,7705483,346380,True,5035,cardiovascular|shock / hypotension|hypotension,"458.9, I95.9",Major


## 1.2 Demographics & stay metadata

In [14]:
# ── Age
# eICU stores age as a string; '>89' patients are de-identified → recode to 90
eicu_pt['age_clean'] = (
    eicu_pt['age']
    .astype(str)
    .str.replace('> 89', '90', regex=False)
    .str.replace('>89',  '90', regex=False)
    .str.strip()
)
eicu_pt['age_clean'] = pd.to_numeric(eicu_pt['age_clean'], errors='coerce')

# ── Gender
eicu_pt['gender_binary'] = eicu_pt['gender'].map({'Male': 1, 'Female': 0})

# ── Ethnicity → 5 harmonized groups
ETH_MAP_EICU = {
    'Caucasian':              'White',
    'African American':       'African',
    'Hispanic':               'Hispanic',
    'Asian':                  'Asian',
    'Native American':        'American',
    'Other/Unknown':          'Unknown',
    '':                       'Unknown',
}
eicu_pt['ethnicity_grp'] = (
    eicu_pt['ethnicity']
    .fillna('Other/Unknown')
    .map(lambda x: ETH_MAP_EICU.get(x, 'Other'))
)

# ── ICU LOS (days)
# unitdischargeoffset is in minutes from ICU admission
eicu_pt['icu_los_days'] = eicu_pt['unitdischargeoffset'] / 1440
eicu_pt.loc[eicu_pt['icu_los_days'] < 0, 'icu_los_days'] = np.nan  # sanity

# ── Unit type → harmonized care-unit category
UNIT_MAP_EICU = {
    'MICU':          'MICU',
    'CCU-CTICU':     'CCU',
    'Cardiac ICU':   'CCU',
    'CSICU':         'CSICU',
    'SICU':          'SICU',
    'NEURO SICU':    'NSICU',
    'Med-Surg ICU':  'MICU',
}
eicu_pt['careunit'] = eicu_pt['unittype'].map(lambda x: UNIT_MAP_EICU.get(str(x), 'Other'))

# ── Admission source
ADMIT_SRC_EICU = {
    'Emergency Department': 'Emergency',
    'Direct Admit':         'Direct',
    'Floor':                'Floor',
    'Operating Room':       'Operating Room',
    'Recovery Room':        'Recovery Room',
    'Other ICU':            'Transfer',
    'Other Hospital':       'Transfer',
    'Chest Pain Center':    'Chest Pain',
}
eicu_pt['admit_source'] = eicu_pt['hospitaladmitsource'].map(
    lambda x: ADMIT_SRC_EICU.get(str(x), 'Unknown')
)

# ── Mortality label
eicu_pt['mortality'] = (eicu_pt['unitdischargestatus'] == 'Expired').astype(int)

print('eICU demographics processed.')
print(eicu_pt[['age_clean','gender_binary','ethnicity_grp','icu_los_days',
               'careunit','admit_source','mortality']].describe())

eICU demographics processed.
       age_clean  gender_binary  icu_los_days  mortality
count   2516.000       2516.000      2520.000   2520.000
mean      63.281          0.599         2.419      0.050
std       17.765          0.490         3.456      0.218
min       15.000          0.000         0.000      0.000
25%       53.000          0.000         0.790      0.000
50%       66.000          1.000         1.472      0.000
75%       77.000          1.000         2.777      0.000
max       90.000          1.000        46.180      1.000


## 1.3 First-24h lab features

In [15]:
# labresultoffset: minutes from ICU admission (negative = before admission)
# Keep only first 24 hours (0–1440 min) to match MIMIC's first-24h window

LAB_TARGETS = {
    # eICU label            : harmonized column prefix
    'creatinine':            'creatinine',
    'BUN':                   'bun',
    'sodium':                'sodium',
    'potassium':             'potassium',
    'glucose':               'glucose',
    'Hgb':                   'hemoglobin',
    'Hct':                   'hematocrit',
    'WBC x 1000':            'wbc',
    'platelets x 1000':      'platelets',
    'bicarbonate':           'bicarbonate',
    'ALT (SGPT)':            'alt',
    'albumin':               'albumin',
    'lactate':               'lactate',
    'PT - INR':              'inr',
}

lab24 = eicu_lab[
    (eicu_lab['labresultoffset'] >= 0) &
    (eicu_lab['labresultoffset'] <= 1440) &
    (eicu_lab['labname'].isin(LAB_TARGETS.keys()))
].copy()

lab24['lab_col'] = lab24['labname'].map(LAB_TARGETS)
lab24['labresult'] = pd.to_numeric(lab24['labresult'], errors='coerce')

# Aggregate: min, max, mean per stay per lab
lab_agg = (
    lab24
    .groupby(['patientunitstayid', 'lab_col'])['labresult']
    .agg(['min', 'max', 'mean'])
    .reset_index()
)
lab_agg.columns = ['patientunitstayid', 'lab_col', 'lab_min', 'lab_max', 'lab_mean']

# Pivot to wide format: one row per stay
lab_wide = lab_agg.pivot_table(
    index='patientunitstayid',
    columns='lab_col',
    values=['lab_min', 'lab_max', 'lab_mean']
)
# Flatten column multi-index: e.g. ('lab_min', 'creatinine') → 'creatinine_min'
lab_wide.columns = [f'{col}_{stat.replace("lab_","")}' for stat, col in lab_wide.columns]
lab_wide = lab_wide.reset_index()

print(f'Lab feature shape: {lab_wide.shape}')
print(lab_wide.columns.tolist()[:10], '...')

Lab feature shape: (2208, 43)
['patientunitstayid', 'albumin_max', 'alt_max', 'bicarbonate_max', 'bun_max', 'creatinine_max', 'glucose_max', 'hematocrit_max', 'hemoglobin_max', 'inr_max'] ...


## 1.4 First-24h vital sign features

In [16]:
# vitalperiodicoffset = minutes from ICU admission
VITAL_COLS = {
    'heartrate':         'heartrate',
    'respiration':       'respiration',
    'sao2':              'sao2',
    'temperature':       'temp',
    'systemicsystolic':  'ssystolic',
    'systemicdiastolic': 'sdiastolic',
    'systemicmean':      'systemicmean',
}

vp24 = eicu_vp[
    (eicu_vp['observationoffset'] >= 0) &
    (eicu_vp['observationoffset'] <= 1440)
].copy()

vital_agg_frames = []
for raw_col, prefix in VITAL_COLS.items():
    if raw_col not in vp24.columns:
        print(f'  WARNING: {raw_col} not found in vitalPeriodic — skipping')
        continue
    tmp = (
        vp24[['patientunitstayid', raw_col]]
        .replace(-1, np.nan)          # eICU uses -1 as missing sentinel
        .dropna(subset=[raw_col])
        .groupby('patientunitstayid')[raw_col]
        .agg(['min','max','mean'])
        .rename(columns={'min': f'{prefix}_min', 'max': f'{prefix}_max', 'mean': f'{prefix}_mean'})
    )
    vital_agg_frames.append(tmp)

vital_wide = pd.concat(vital_agg_frames, axis=1).reset_index()
print(f'Vital feature shape: {vital_wide.shape}')
print(vital_wide.columns.tolist())

Vital feature shape: (2370, 22)
['patientunitstayid', 'heartrate_min', 'heartrate_max', 'heartrate_mean', 'respiration_min', 'respiration_max', 'respiration_mean', 'sao2_min', 'sao2_max', 'sao2_mean', 'temp_min', 'temp_max', 'temp_mean', 'ssystolic_min', 'ssystolic_max', 'ssystolic_mean', 'sdiastolic_min', 'sdiastolic_max', 'sdiastolic_mean', 'systemicmean_min', 'systemicmean_max', 'systemicmean_mean']


## 1.5 Diagnosis ICD-9





In [17]:
def icd9_to_chapter(code):
    """Map ICD-9 code string to chapter name (broad disease category)."""
    try:
        # Remove decimal and leading zeros; handle V/E codes
        c = str(code).strip().upper().replace('.', '')
        if c.startswith('V'):
            return 'Supplementary'
        if c.startswith('E'):
            return 'External causes'
        n = int(c[:3])
    except (ValueError, TypeError):
        return 'Unknown'

    chapters = [
        (1,   139,  'Infectious & parasitic'),
        (140, 239,  'Neoplasms'),
        (240, 279,  'Endocrine & metabolic'),
        (280, 289,  'Blood disorders'),
        (290, 319,  'Mental disorders'),
        (320, 389,  'Nervous system'),
        (390, 459,  'Circulatory'),
        (460, 519,  'Respiratory'),
        (520, 579,  'Digestive'),
        (580, 629,  'Genitourinary'),
        (630, 679,  'Pregnancy/childbirth'),
        (680, 709,  'Skin'),
        (710, 739,  'Musculoskeletal'),
        (740, 759,  'Congenital'),
        (760, 779,  'Perinatal'),
        (780, 799,  'Symptoms & signs'),
        (800, 999,  'Injury & poisoning'),
    ]
    for lo, hi, name in chapters:
        if lo <= n <= hi:
            return name
    return 'Unknown'

# eICU diagnosis table has 'icd9code' column
# Take the primary diagnosis (diagnosispriority == 'Primary' or lowest sequence number)
eicu_dx['icd9code'] = eicu_dx['icd9code'].astype(str).str.strip()

# Keep primary diagnosis per stay (diagnosispriority = 'Primary' if available)
primary_dx = (
    eicu_dx[eicu_dx['diagnosispriority'].str.lower().str.contains('primary', na=False)]
    if 'diagnosispriority' in eicu_dx.columns
    else eicu_dx.sort_values('diagnosisid').groupby('patientunitstayid').first().reset_index()
)

dx_first = (
    primary_dx
    .sort_values('diagnosisid' if 'diagnosisid' in primary_dx.columns else primary_dx.columns[0])
    .groupby('patientunitstayid')['icd9code']
    .first()
    .reset_index()
)
dx_first['icd9_chapter'] = dx_first['icd9code'].apply(icd9_to_chapter)
dx_first = dx_first[['patientunitstayid', 'icd9_chapter']]

print(dx_first['icd9_chapter'].value_counts())

icd9_chapter
Circulatory               400
Unknown                   327
Respiratory               250
Symptoms & signs          147
Injury & poisoning        135
Digestive                 122
Endocrine & metabolic     117
Infectious & parasitic     93
Genitourinary              32
Mental disorders           23
Nervous system             22
Neoplasms                  17
External causes             9
Blood disorders             6
Skin                        5
Supplementary               2
Pregnancy/childbirth        2
Musculoskeletal             2
Name: count, dtype: int64


## 1.6 Merge all eICU features

In [18]:
# Base: demographics + stay metadata + label
DEMO_COLS = [
    'patientunitstayid',
    'age_clean', 'gender_binary', 'ethnicity_grp',
    'icu_los_days', 'careunit', 'admit_source',
    'mortality'
]
eicu_base = eicu_pt[DEMO_COLS].copy()

eicu_features = (
    eicu_base
    .merge(lab_wide,   on='patientunitstayid', how='left')
    .merge(vital_wide, on='patientunitstayid', how='left')
    .merge(dx_first,   on='patientunitstayid', how='left')
)

# Rename id column for clarity
eicu_features.rename(columns={'patientunitstayid': 'stay_id'}, inplace=True)
eicu_features['dataset'] = 'eicu'

print(f'eICU feature table: {eicu_features.shape}')
print(f'Mortality rate: {eicu_features["mortality"].mean():.3f}')
eicu_features.head(3)

eICU feature table: (2520, 73)
Mortality rate: 0.050


,stay_id,age_clean,gender_binary,ethnicity_grp,icu_los_days,careunit,admit_source,mortality,albumin_max,alt_max,bicarbonate_max,bun_max,creatinine_max,glucose_max,hematocrit_max,hemoglobin_max,inr_max,lactate_max,platelets_max,potassium_max,sodium_max,wbc_max,albumin_mean,alt_mean,bicarbonate_mean,...,sodium_min,wbc_min,heartrate_min,heartrate_max,heartrate_mean,respiration_min,respiration_max,respiration_mean,sao2_min,sao2_max,sao2_mean,temp_min,temp_max,temp_mean,ssystolic_min,ssystolic_max,ssystolic_mean,sdiastolic_min,sdiastolic_max,sdiastolic_mean,systemicmean_min,systemicmean_max,systemicmean_mean,icd9_chapter,dataset
0,141764,87.000,0.000,White,0.239,MICU,Unknown,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,90.000,138.000,106.652,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,eicu
1,141765,87.000,0.000,White,1.562,MICU,Unknown,0,NaN,NaN,21.000,28.000,1.040,61.000,37.800,12.300,NaN,NaN,191.000,4.100,139.000,10.200,NaN,NaN,21.000,...,139.000,10.200,72.000,116.000,83.191,17.000,39.000,24.783,95.000,98.000,96.609,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,eicu
2,143870,76.000,1.000,White,0.551,SICU,Operating Room,0,NaN,NaN,25.000,14.000,1.140,140.000,34.100,12.300,NaN,NaN,152.000,4.400,139.000,11.700,NaN,NaN,23.500,...,133.000,11.700,40.000,55.000,45.449,45.000,86.000,67.032,80.000,100.000,96.285,NaN,NaN,NaN,53.000,138.000,109.816,36.000,57.000,42.715,47.000,79.000,60.835,Unknown,eicu


In [19]:
out_path_2 = os.path.join(EICU_OUT_DIR, 'eicu_features.csv')
eicu_features.to_csv(out_path_2, index=False)
print(f'Saved: {out_path_2}')
print(f'Shape: {eicu_features.shape[0]:,} rows × {eicu_features.shape[1]} cols')

Saved: /content/drive/MyDrive/AI in Medicine/data/output_data/eicu_train/eicu_features.csv
Shape: 2,520 rows × 73 cols
